# Neural Network Model

In [1]:
# import relevant libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import LabelEncoder

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam

from tqdm.auto import tqdm

In [2]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Install a CUDA-enabled PyTorch build or enable GPU.")

device = torch.device("cuda")
device
torch.__version__

'2.9.0+cu126'

In training the neural network, TF-IDF of the word and character n-grams will be used as the feature to be fed throught the model. As mentioned before in the corpus analysis notebook, TF-IDF can be used as a metric to be used for classifying text authorship. For the model architecture, both shallow and deep neural network architecture will be explored. We are curious to see if a simpler architecture is more fit considering the small dataset, and the limited authors we have. Prior works have tried to use both architectures for the models [1,2,3].

The model will utilize ReLU activation functions to improve training efficiency and mitigate vanishing gradient issues [4]. In addition, dropout regularization will be introduced to reduce overfitting [5] and over reliance on specific neurons. The Adam optimizer will be used due to its adaptive learning rate and strong empirical performance across a wide range of machine learning tasks [6] Additionally, early stopping is applied to prevent overfitting by monitoring validation performance during training [7].

## Data prep and TF-IDF features

In [3]:
DATA_PATH = Path("llm_data/llm-dataset/eli5_all_llm_answers_cleaned.csv")
df = pd.read_csv(DATA_PATH)


long_df = df.melt(
    id_vars = ["q_id", "question"],
    value_vars = ["chatgpt", "deepseek", "gemini"],
    var_name = "author",
    value_name = "text",
).dropna(subset = ["text"])
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(long_df["author"])
texts = long_df["text"].astype(str).tolist()


### Data Split

In [ ]:
# 70-15-15 split for train, validation, and test sets
X_train_texts, X_temp_texts, y_train, y_temp = train_test_split(
    texts, 
    labels, 
    test_size = 0.3, 
    random_state = 67, 
    stratify = labels
)
X_val_texts, X_test_texts, y_val, y_test = train_test_split(
    X_temp_texts, 
    y_temp, 
    test_size = 0.5, 
    random_state = 67, 
    stratify = y_temp
)

In [20]:
# Inspect training data and labels
train_df = pd.DataFrame({"text": X_train_texts, "label_id": y_train})
train_df["label_name"] = label_encoder.inverse_transform(train_df["label_id"])

print("Training label distribution:")
print(train_df["label_name"].value_counts())

print("\nSample training rows:")
display(train_df.head(20))

Training label distribution:
label_name
chatgpt     7000
deepseek    7000
gemini      7000
Name: count, dtype: int64

Sample training rows:


,text,label_id,label_name
0,"Okay! So, imagine you have a big box of toys t...",0,chatgpt
1,"Okay, imagine your eyes are like tiny cameras ...",0,chatgpt
2,Think of the store like a piggy bank that you ...,1,deepseek
3,"When something very, very sad happens outside ...",2,gemini
4,"Okay, imagine your tongue is like a little mag...",0,chatgpt
5,Dwarf Fortress is like a big video game where ...,0,chatgpt
6,We're looking at the very edge of the universe...,1,deepseek
7,"Okay, imagine the sky is like a big sponge tha...",0,chatgpt
8,Circuit boards are often green because of the ...,1,deepseek
9,Think of cancer like a bunch of bad weeds in a...,1,deepseek


In [27]:
# Inspect validation data and labels
val_df = pd.DataFrame({"text": X_val_texts, "label_id": y_val})
val_df["label_name"] = label_encoder.inverse_transform(val_df["label_id"])

print("Validation label distribution:")
print(val_df["label_name"].value_counts())

print("\nSample validation rows:")
display(val_df.sample(min(5, len(val_df)), random_state=67))

print(val_df.sample(min(5, len(val_df)), random_state=67))

Validation label distribution:
label_name
chatgpt     1500
deepseek    1500
gemini      1500
Name: count, dtype: int64

Sample validation rows:


,text,label_id,label_name
4089,"Imagine you really, really want a new slide at...",2,gemini
1391,"Imagine your body is like a big water balloon,...",2,gemini
1455,Sloths move slowly because it helps them hide ...,1,deepseek
4311,"Okay, so imagine your body is like a big city ...",0,chatgpt
4187,"Okay, imagine you're a giant, super-duper big!...",2,gemini


                                                   text  label_id label_name
4089  Imagine you really, really want a new slide at...         2     gemini
1391  Imagine your body is like a big water balloon,...         2     gemini
1455  Sloths move slowly because it helps them hide ...         1   deepseek
4311  Okay, so imagine your body is like a big city ...         0    chatgpt
4187  Okay, imagine you're a giant, super-duper big!...         2     gemini


### Getting the n-grams

In [ ]:
word_vectorizer = TfidfVectorizer(
    analyzer="word", 
    ngram_range = (2, 3),  # word bigram and trigram 
    min_df = 5, 
    max_features = 32768, # limit for computational efficiency
    dtype = np.float32,
    )

char_vectorizer = TfidfVectorizer(
    analyzer="char", 
    ngram_range = (3, 5), # char 3-grams to 5-grams
    min_df = 5, 
    max_features = 32768,  # limit for computational efficiency
    dtype = np.float32,
    )

vectorizer = FeatureUnion(
    [("word_tfidf", word_vectorizer), ("char_tfidf", char_vectorizer)]
)

X_train = vectorizer.fit_transform(X_train_texts)
X_val = vectorizer.transform(X_val_texts)
X_test = vectorizer.transform(X_test_texts)

num_features = X_train.shape[1]
num_classes = len(label_encoder.classes_)

## Grid search for hidden layer sizes

In [ ]:

class LLMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_units, num_classes, dropout_rate = 0.3):
        super().__init__()
        layers_list = []
        prev_dim = input_dim
        for units in hidden_units:
            layers_list.append(nn.Linear(prev_dim, units))
            layers_list.append(nn.ReLU())
            layers_list.append(nn.Dropout(dropout_rate))
            prev_dim = units
        layers_list.append(nn.Linear(prev_dim, num_classes))
        self.model = nn.Sequential(*layers_list)

    def forward(self, x):
        return self.model(x)

def make_dataloader(features, labels, batch_size, shuffle = False):
    features_tensor = torch.tensor(features, dtype = torch.float32)
    labels_tensor = torch.tensor(labels, dtype = torch.long)
    dataset = TensorDataset(features_tensor, labels_tensor)
    return DataLoader(dataset, batch_size = batch_size, shuffle = shuffle)

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = loss_fn(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_x.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            logits = model(batch_x)
            loss = loss_fn(logits, batch_y)
            total_loss += loss.item() * batch_x.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            total += batch_x.size(0)
    return total_loss / total, correct / total

def fit_model(model, train_loader, val_loader, epochs, lr, patience, device):
    optimizer = Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    best_state = None
    best_val_loss = float("inf")
    patience_left = patience
    history = []
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })
        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left == 0:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return history

X_train_dense = X_train.toarray().astype(np.float32)
X_val_dense = X_val.toarray().astype(np.float32)
X_test_dense = X_test.toarray().astype(np.float32)

train_loader = make_dataloader(X_train_dense, y_train, batch_size=16, shuffle=True)
val_loader = make_dataloader(X_val_dense, y_val, batch_size=16)
test_loader = make_dataloader(X_test_dense, y_test, batch_size=16)

In [7]:
hidden_search_space = [
    (32,),
    (64,),
    (128,),
    (64, 32), 
    (128, 64),
    (32, 32, 32),
    (64, 64, 64),
 ]
lr_search_space = [1e-4, 5e-5, 2e-4]

best_config = None
best_val_acc = 0.0
results = []
histories = []

input_dim = X_train.shape[1]

for hidden_units in hidden_search_space:
    for lr in lr_search_space:
        print(f"\nTraining model with hidden_units={hidden_units}, lr={lr}")
        model = LLMClassifier(
            input_dim = input_dim,
            hidden_units = hidden_units,
            num_classes = num_classes,
            dropout_rate = 0.5,
        ).to(device)
        
        history = fit_model(
            model,
            train_loader,
            val_loader,
            epochs = 40,
            lr = lr,
            patience = 2,
            device = device,
        )
        val_acc = history[-1]["val_acc"] if history else 0.0
        results.append({"hidden_units": hidden_units, "lr": lr, "val_acc": val_acc})
        histories.append({"hidden_units": hidden_units, "lr": lr, "history": history})
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_config = {"hidden_units": hidden_units, "lr": lr}

best_config, best_val_acc


Training model with hidden_units=(32,), lr=0.0001
Epoch 01 | train_loss=0.9662 train_acc=0.6683 val_loss=0.7858 val_acc=0.9236
Epoch 02 | train_loss=0.6324 train_acc=0.8972 val_loss=0.4886 val_acc=0.9564
Epoch 03 | train_loss=0.4192 train_acc=0.9243 val_loss=0.3264 val_acc=0.9613
Epoch 04 | train_loss=0.3014 train_acc=0.9429 val_loss=0.2362 val_acc=0.9660
Epoch 05 | train_loss=0.2267 train_acc=0.9567 val_loss=0.1820 val_acc=0.9702
Epoch 06 | train_loss=0.1788 train_acc=0.9657 val_loss=0.1471 val_acc=0.9736
Epoch 07 | train_loss=0.1459 train_acc=0.9714 val_loss=0.1231 val_acc=0.9731
Epoch 08 | train_loss=0.1202 train_acc=0.9782 val_loss=0.1053 val_acc=0.9749
Epoch 09 | train_loss=0.1007 train_acc=0.9812 val_loss=0.0925 val_acc=0.9769
Epoch 10 | train_loss=0.0844 train_acc=0.9863 val_loss=0.0828 val_acc=0.9778
Epoch 11 | train_loss=0.0704 train_acc=0.9888 val_loss=0.0756 val_acc=0.9784
Epoch 12 | train_loss=0.0597 train_acc=0.9907 val_loss=0.0696 val_acc=0.9793
Epoch 13 | train_loss=0.0

({'hidden_units': (128,), 'lr': 0.0001}, 0.9833333333333333)

## Graphs of Loss and Accura

In [13]:
import ipywidgets as widgets
from IPython.display import display

lrs = sorted({item["lr"] for item in histories})
hidden_units_list = sorted({item["hidden_units"] for item in histories})

lr_dropdown = widgets.Dropdown(options=lrs, description="LR:")
hidden_dropdown = widgets.Dropdown(options=hidden_units_list, description="Hidden:")

def plot_history(lr, hidden_units):
    match = None
    for item in histories:
        if item["lr"] == lr and item["hidden_units"] == hidden_units:
            match = item
            break
    if match is None or not match["history"]:
        print("No history found for this configuration.")
        return
    history = match["history"]
    epochs = [h["epoch"] for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss = [h["val_loss"] for h in history]
    train_acc = [h["train_acc"] for h in history]
    val_acc = [h["val_acc"] for h in history]

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_loss, label="train")
    plt.plot(epochs, val_loss, linestyle="--", label="val")
    plt.title("Loss per epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(fontsize=8)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_acc, label="train")
    plt.plot(epochs, val_acc, linestyle="--", label="val")
    plt.title("Accuracy per epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend(fontsize=8)

    plt.suptitle(f"Learning rate = {lr}, hidden units = {hidden_units}")
    plt.tight_layout()
    plt.show()

ui = widgets.HBox([lr_dropdown, hidden_dropdown])
out = widgets.interactive_output(plot_history, {"lr": lr_dropdown, "hidden_units": hidden_dropdown})
display(ui, out)

Output()

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

best_model = None

def train_best_model():
    model = MLPClassifier(
        input_dim=input_dim,
        hidden_units=best_config["hidden_units"],
        num_classes=num_classes,
        dropout_rate=0.5,
    ).to(device)
    fit_model(
        model,
        train_loader,
        val_loader,
        epochs=40,
        lr=best_config["lr"],
        patience=2,
        device=device,
    )
    return model

def ensure_best_model():
    global best_model
    if best_config is None:
        raise RuntimeError("best_config is None. Run the grid search cell first.")
    if best_model is None:
        print("Training best model with best_config...")
        best_model = train_best_model()
    return best_model

text_input = widgets.Textarea(
    value="",
    placeholder="Paste text here...",
    description="Text:",
    layout=widgets.Layout(width="100%", height="120px"),
)
predict_button = widgets.Button(description="Predict", button_style="primary")
output = widgets.Output()

def on_predict(_):
    with output:
        clear_output()
        text = text_input.value.strip()
        if not text:
            print("Paste some text to classify.")
            return
        model = ensure_best_model()
        model.eval()
        features = vectorizer.transform([text]).toarray().astype(np.float32)
        x = torch.tensor(features, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits = model(x)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = int(np.argmax(probs))
        pred_label = label_encoder.inverse_transform([pred_idx])[0]
        print(f"Predicted author: {pred_label}\n")
        for label, prob in zip(label_encoder.classes_, probs):
            print(f"{label}: {prob:.4f}")

predict_button.on_click(on_predict)

display(widgets.VBox([text_input, predict_button, output]))

[1]
Modupe, A., Celik, T., Marivate, V., & Olugbara, O. O. (2022). Post-authorship attribution using regularized deep neural network. Applied Sciences, 12(15), 7518.

[2]
Saha, N., Das, P., & Saha, H. N. (2018). Authorship attribution of short texts using multi-layer perceptron. International Journal of Applied Pattern Recognition, 5(3), 251-259.

[3]
Sari, Y., Vlachos, A., & Stevenson, M. (2017, April). Continuous n-gram representations for authorship attribution. In Proceedings of the 15th conference of the European chapter of the association for computational linguistics: Volume 2, short papers (pp. 267-273).

[4]
GeeksforGeeks. (2025, March 11). ReLU activation function in deep learning. https://www.geeksforgeeks.org/deep-learning/relu-activation-function-in-deep-learning/

[5]
Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., & Salakhutdinov, R. (2014). Dropout: a simple way to prevent neural networks from overfitting. The journal of machine learning research, 15(1), 1929-1958.

[6]
Kingma, D. P., & Ba, J. (2014). Adam: A method for stochastic optimization. arXiv preprint arXiv:1412.6980.

[7]
GeeksforGeeks. (2025, August 28). Regularization by early stopping. https://www.geeksforgeeks.org/machine-learning/regularization-by-early-stopping/